# 1. Librairies

In [1]:
import pandas as pd
import numpy as np

In [2]:
from pathlib import Path

In [3]:
pd.set_option('display.max_columns', None)

#### Chemin d'acces

In [4]:
# Racine du projet : le dossier parent du dossier notebook
BASE_DIR = Path.cwd().parent

# Dossiers de données
DATA_DIR = BASE_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
CLEAN_DIR = DATA_DIR / "clean"

# Fichiers
RAW_FILE = RAW_DIR / "churn.csv"
CLEAN_FILE = CLEAN_DIR / "churn_clean.parquet"

# 2. Lecture

In [5]:
df = pd.read_csv(RAW_FILE)

initial = df.shape
print(f"Dimensions : {initial}")
df.head()

Dimensions : (5150, 8)


,client_id,age,sexe,anciennete_mois,montant_mensuel_fcfa,nb_reclamations,type_contrat,churn
0,14453,43.0,Femme,53.0,18800.0,2,Mensuel,0
1,10434,24.0,Femme,28.0,7200.0,0,Annuel,0
2,12153,63.0,Homme,51.0,6600.0,2,Mensuel,0
3,14341,20.0,Femme,13.0,16600.0,0,Mensuel,0
4,12905,52.0,Femme,67.0,16000.0,3,Mensuel,0


In [6]:
print(f"{' '*40}{'=' * 60}")
print(f"{' '*60} Infos sur les données")
print(df.info())

print()

print(f"{' '*40}{'=' * 60}")
print(f"{' '*60} Types de données")
print(df.dtypes)

print()

print(f"{' '*40}{'=' * 60}")
print(f"{' '*50} Nombre de valeurs unique par colonne")
print(df.nunique())

                                                             Infos sur les données
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5150 entries, 0 to 5149
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   client_id             5150 non-null   int64  
 1   age                   4894 non-null   float64
 2   sexe                  5001 non-null   object 
 3   anciennete_mois       4990 non-null   float64
 4   montant_mensuel_fcfa  4862 non-null   float64
 5   nb_reclamations       5150 non-null   int64  
 6   type_contrat          4931 non-null   object 
 7   churn                 5150 non-null   int64  
dtypes: float64(3), int64(3), object(2)
memory usage: 322.0+ KB
None

                                                             Types de données
client_id                 int64
age                     float64
sexe                     object
anciennete_mois         float64
montant_mensuel_fcfa    f

#### Description

In [7]:
df.describe()

,client_id,age,anciennete_mois,montant_mensuel_fcfa,nb_reclamations,churn
count,5150.000000,4894.000000,4990.000000,4862.000000,5150.000000,5150.000000
mean,12503.327184,43.898243,36.280762,14865.919375,1.222524,0.208738
std,1441.426630,15.364814,20.693749,5902.695224,1.094042,0.406446
min,10001.000000,18.000000,1.000000,3000.000000,0.000000,0.000000
25%,11253.250000,30.000000,19.000000,10800.000000,0.000000,0.000000
50%,12512.500000,44.000000,36.000000,14700.000000,1.000000,0.000000
75%,13748.750000,57.000000,54.000000,18800.000000,2.000000,0.000000
max,15000.000000,70.000000,72.000000,39900.000000,7.000000,1.000000


#### Valeurs manquantes

In [8]:
manquants = pd.DataFrame({
    'nbre' : df.isna().sum(),
    'pct' : df.isna().mean().round(2)
})

manquants.sort_values(by='nbre', ascending=False)

,nbre,pct
montant_mensuel_fcfa,288,0.06
age,256,0.05
type_contrat,219,0.04
anciennete_mois,160,0.03
sexe,149,0.03
client_id,0,0.00
nb_reclamations,0,0.00
churn,0,0.00


#### Valeurs dupliquées

In [9]:
print(f"Nombre dupliqué : {df.duplicated().sum()}")

Nombre dupliqué : 150


# 3. Nettoyage

In [10]:
df.dropna(inplace=True)
df.drop_duplicates(inplace=True)

In [11]:
manquants = pd.DataFrame({
    'nbre' : df.isna().sum(),
    'pct' : df.isna().mean().round(2)
})

print(manquants.sort_values(by='nbre', ascending=False).head(2))
print(f"\nNombre dupliqué(verif) : {df.duplicated().sum()}")

           nbre  pct
client_id     0  0.0
age           0  0.0

Nombre dupliqué(verif) : 0


#### Suppression de id_client

Apres nettoyage des `doublons`, nous n'avons plus besoin de l'id des clients.

In [12]:
df.drop('client_id', axis=1, inplace=True)

inter = df.shape
print(f"Dim Avant : {initial} | Dim Apres : {inter}")

Dim Avant : (5150, 8) | Dim Apres : (4052, 7)


#### Conditions de base

La condition de base s'applique pour que toutes les données numériques de ce dataset soit `superieur à 0`(mise en place malgré la confirmation de la méthode `.describe()`).

In [13]:
df.columns

Index(['age', 'sexe', 'anciennete_mois', 'montant_mensuel_fcfa',
       'nb_reclamations', 'type_contrat', 'churn'],
      dtype='object')

In [14]:
df = df.loc[(df['anciennete_mois'] >= 0) &
    (df['montant_mensuel_fcfa'] >= 0) &
    (df['age'] >= 10) &    # 10 ans ou plus
    (df['nb_reclamations'] >= 0)
]            

final = df.shape
print(f"Dim Avant : {inter} | Dim Apres : {final}")

Dim Avant : (4052, 7) | Dim Apres : (4052, 7)


#### Harmonisation

In [15]:
df['sexe'] = df['sexe'].str.lower()
df['type_contrat'] = df['type_contrat'].str.lower()

In [16]:
df.head()

,age,sexe,anciennete_mois,montant_mensuel_fcfa,nb_reclamations,type_contrat,churn
0,43.0,femme,53.0,18800.0,2,mensuel,0
1,24.0,femme,28.0,7200.0,0,annuel,0
2,63.0,homme,51.0,6600.0,2,mensuel,0
3,20.0,femme,13.0,16600.0,0,mensuel,0
4,52.0,femme,67.0,16000.0,3,mensuel,0


# 4. Sauvegarde

In [17]:
CLEAN_DIR.mkdir(parents=True, exist_ok=True)  # crée le dossier s'il n'existe pas
df.to_parquet(CLEAN_FILE, index=False)